# Prompt Evaluation & Optimization with Vertex AI SDK

# ----------------------------------------------------

This notebook demonstrates how to perform prompt evaluation and optimization
using Vertex AI Generative AI Evaluation SDK.
It includes: dataset prep, metric setup, evaluation run, optimization loop.

### Evaluation Dimensions

1. Functional Performance: How accurate/relevant/correct the generated content is
    1. Mathematical Metrics: BLEU, ROUGUE, METEOR, Exactness
    2. Metrics with LLM as a judge: Relevancy, Conciseness, Correctness, Specificity

2. Operational Performance: How efficient generation is
    - Latency, Token Count/Cost
    
3. Utility Performance: how easy it is to consume the generated content
    - Readability: ARI Grade, Flesch Kincard Score:  Metrics (it focuses on ensuring sentences have lower number of words, and words have lower number of letters)


## Setup and installation


In [1]:
!pip install --upgrade google-cloud-aiplatform[evaluation] --quiet

In [2]:
import os
import pandas as pd
from google.cloud import aiplatform
import vertexai
from vertexai.evaluation import EvalTask, PointwiseMetric, PointwiseMetricPromptTemplate

In [3]:
PROJECT_ID = "bdc-trainings"
LOCATION = "us-central1"
vertexai.init(project=PROJECT_ID, location=LOCATION)
print(f"Initialized Vertex AI for project {PROJECT_ID}, location {LOCATION}")

Initialized Vertex AI for project bdc-trainings, location us-central1


## Prepare evaluation dataset


In [4]:
df = pd.DataFrame({
    'prompt': ["Summarize the benefits of AI in healthcare.", "Explain cloud computing in simple terms."],
    'response': ["AI improves diagnostics and personalized care.", "Cloud computing means using remote servers for storage and processing."],
    'reference': ["AI in healthcare aids diagnosis and treatment personalization.", "Cloud computing uses internet-based servers instead of local ones."]
})

df.to_json("eval_dataset.jsonl", orient="records", lines=True)
print("Evaluation dataset created and saved as eval_dataset.jsonl")


Evaluation dataset created and saved as eval_dataset.jsonl


## Define evaluation metrics


In [5]:
readability_metric = PointwiseMetric(
    metric="readability_grade_level",
    metric_prompt_template=PointwiseMetricPromptTemplate(
        criteria={
            "grade_level": (
                "Estimate the U.S. grade-level readability of the response. Lower is simpler, higher is complex."
            ),
            "conciseness": (
                "Evaluate how concise the response is: avoids unnecessary words while preserving meaning."
            )
        },
        rating_rubric={
            "1": "Excellent readability and conciseness.",
            "0": "Average readability and conciseness.",
            "-1": "Poor readability or verbosity."
        },
        input_variables=["prompt", "response", "reference"]
    )
)
print("Custom readability metric defined.")


Custom readability metric defined.


## Run evaluation

In [7]:
eval_task = EvalTask(
    dataset="eval_dataset.jsonl",
    metrics=[readability_metric, "bleu", "rouge_l_sum"],
    experiment="prompt-eval-experiment"
)

print("Running evaluation task...")
eval_result = eval_task.evaluate(
    model="projects/bdc-trainings/locations/us-central1/models/gemini-2.5-flash",
    prompt_template="{prompt}"
)
print("Evaluation completed.")


Running evaluation task...


Associating projects/456822750436/locations/us-central1/metadataStores/default/contexts/prompt-eval-experiment-dceca736-9447-4275-95b7-5b6470c7b4a6 to Experiment: prompt-eval-experiment


Logging Eval Experiment metadata: {'prompt_template': '{prompt}', 'model_name': 'projects/bdc-trainings/locations/us-central1/models/gemini-2.5-flash'}
Assembling prompts from the `prompt_template`. The `prompt` column in the `EvalResult.metrics_table` has the assembled prompts used for model response generation.


ValueError: The `model` parameter or `baseline_model` in pairwise metric is specified, but the evaluation `dataset` contains model response column or baseline model response column `response` to perform bring-your-own-response(BYOR) evaluation. If you would like to perform evaluation using the dataset with the existing model response column or or baseline model response column `response`, please remove `model` parameter in `EvalTask.evaluate()` function or `baseline_model` in `PairwiseMetric`.

## View evaluation results


In [ ]:
print(eval_result.metrics_summary)
per_instance = eval_result.to_dataframe()
per_instance.sort_values(by="readability_grade_level", ascending=True).head(5)

# Prompt Optimization


In [4]:
from vertexai.preview.generative_models import PromptOptimizer
optimizer = PromptOptimizer()

optimized_prompt = optimizer.optimize(
    model="gemini-1.5-pro",
    task_type="text-generation",
    prompt="Explain cloud computing in simple terms.",
    optimization_goal="improve conciseness and clarity"
)
print("Optimized prompt:", optimized_prompt.optimized_prompt_text)


In [ ]:
# Re-evaluate optimized prompt
optimized_data = pd.DataFrame({
    'prompt': [optimized_prompt.optimized_prompt_text],
    'response': ["Cloud computing is using online servers to store and process data instead of your computer."],
    'reference': ["Cloud computing uses internet-based servers instead of local ones."]
})
optimized_data.to_json("eval_optimized.jsonl", orient="records", lines=True)


In [ ]:
re_eval = EvalTask(
    dataset="eval_optimized.jsonl",
    metrics=[readability_metric, "bleu", "rouge_l_sum"],
    experiment="prompt_re_eval_experiment"
)


In [ ]:
optimized_result = re_eval.evaluate(
    model="projects/your-project/locations/us-central1/models/YOUR_MODEL_ID",
    prompt_template="{prompt}"
)
print("Re-evaluation completed. Metrics:")
print(optimized_result.metrics_summary)
